# Colorado Healthcare Desert Index
## Computing a Composite Access Score for Colorado Census Tracts

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pg8000

%matplotlib inline

In [2]:
def load_env(path: str) -> None:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing .env file at: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())


def get_conn():
    return pg8000.connect(
        host=os.environ["DB_HOST"],
        port=int(os.environ["DB_PORT"]),
        database=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
    )

## Step 1: Load Data from Database

In [3]:
conn = get_conn()
cur = conn.cursor()

cur.execute("""
    SELECT payload
    FROM health_raw.hdi_rows
    WHERE dataset_id = 'acs5_2022_tract_insurance_poverty'
""")

rows = cur.fetchall()
df_insurance = pd.DataFrame([r[0] for r in rows])

print(f"Rows: {len(df_insurance)}")
print(df_insurance.head())

Rows: 1447
  acs_year state_fips county_fips  tract_geoid  poverty_rate  uninsured_rate  \
0     2022         08       08001  08001007801        0.2384          0.1682   
1     2022         08       08001  08001007802        0.3024          0.3327   
2     2022         08       08001  08001007900        0.1748          0.1975   
3     2022         08       08001  08001008000        0.0735          0.1463   
4     2022         08       08001  08001008100        0.3954          0.0247   

   total_uninsured  poverty_universe  total_population  insurance_universe  \
0              654              3850              3889                3889   
1             1447              4349              4359                4349   
2             1207              6126              6126                6112   
3              843              5738              5763                5763   
4               36              1477              1674                1460   

   population_below_poverty  
0        

In [5]:
cur.execute("""
    SELECT payload
    FROM health_raw.hdi_rows
    WHERE dataset_id = 'ahrf_2024-2025_county_physician_supply'
""")

rows = cur.fetchall()
df_physicians = pd.DataFrame([r[0] for r in rows])

print(f"Rows: {len(df_physicians)}")
print(df_physicians.head())

Rows: 64
  state_fips county_fips    county_name ahrf_release  pcp_do_count  \
0         08       08001      Adams, CO    2024-2025            23   
1         08       08003    Alamosa, CO    2024-2025             5   
2         08       08005   Arapahoe, CO    2024-2025            62   
3         08       08007  Archuleta, CO    2024-2025             4   
4         08       08009       Baca, CO    2024-2025             0   

   pcp_md_count  pcp_per_100k  total_pcp_count  total_population  
0           283         56.36              306            542973  
1            10         89.88               15             16689  
2           461         78.42              523            666918  
3            10         99.21               14             14112  
4             2         59.40                2              3367  


In [6]:
cur.execute("""
    SELECT payload
    FROM health_raw.hdi_rows
    WHERE dataset_id = 'tract_to_nearest_ed_2020'
""")

rows = cur.fetchall()
df_distances = pd.DataFrame([r[0] for r in rows])

print(f"Rows: {len(df_distances)}")
print(df_distances.head())

Rows: 1447
   tract_lat   tract_lon county_fips  tract_geoid nearest_ed_ccn  \
0  39.742146 -104.876784       08001  08001007801         060024   
1  39.741862 -104.854956       08001  08001007802         060024   
2  39.747561 -104.874143       08001  08001007900         060024   
3  39.748974 -104.856386       08001  08001008000         060024   
4  39.749147 -104.837083       08001  08001008100         063301   

   nearest_ed_lat  nearest_ed_lon distance_method  haversine_miles  \
0       39.741422     -104.842053       haversine             1.85   
1       39.741422     -104.842053       haversine             0.69   
2       39.741422     -104.842053       haversine             1.76   
3       39.741422     -104.842053       haversine             0.92   
4       39.741348     -104.836910       haversine             0.54   

                             nearest_ed_name  tract_population  \
0  UNIVERSITY OF COLORADO HOSPITAL AUTHORITY              3936   
1  UNIVERSITY OF COLORADO H

## Step 2: Joing Components to Tract Level

In [ ]:
# Physician data is county-level — join to tracts by county_fips.
# Every tract in a county gets that county's physician ratio.
# This is a known limitation: all tracts in Adams County get the same
# physician ratio even though urban tracts likely have better access.
# HRSA doesn't publish physician data below county level for free.
df = df_insurance.merge(
    df_physicians[['county_fips', 'pcp_per_100k']],
    on='county_fips',
    how='left'
)

# Add distance to nearest ED
df = df.merge(
    df_distances[['tract_geoid', 'estimated_drive_minutes', 'haversine_miles']],
    on='tract_geoid',
    how='left'
)

print(f'Joined rows: {len(df)}')
print(f'Null check:')
print(df[['poverty_rate', 'uninsured_rate', 'pcp_per_100k', 'estimated_drive_minutes']].isnull().sum())
print()
print(df[['tract_geoid', 'county_fips', 'poverty_rate', 'uninsured_rate', 'pcp_per_100k', 'estimated_drive_minutes']].head())


## Step 3: Normalize Components to 0–1 Scale

Before we can combine four metrics with different units into a single score,
we need to put them on the same scale. We use **min-max normalization**:

```
normalized = (value - min) / (max - min)
```

This maps every value between 0 and 1, where 0 = the best-access tract in
Colorado and 1 = the worst-access tract.

For **poverty rate** and **uninsured rate**, higher = worse, so we normalize
directly (higher normalized value = more desert-like).

For **physician supply** and **drive time**, we invert after normalizing:
- High physicians per 100k = *good* access → high normalized value → invert to low desert score
- Long drive time = *bad* access → high normalized value → stays as-is

We fill the small number of missing physician values (tracts with no county
match) with the statewide median — a conservative assumption.

In [ ]:
def minmax(series):
    """Min-max normalize a pandas Series to [0, 1]."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return series * 0  # all same value — score everything 0
    return (series - mn) / (mx - mn)

# Fill any missing physician values with statewide median
df['pcp_per_100k'] = df['pcp_per_100k'].fillna(df['pcp_per_100k'].median())

# Normalize each component
# Higher score = MORE desert-like (worse access)
df['poverty_score']    = minmax(df['poverty_rate'])         # high poverty = bad
df['uninsured_score']  = minmax(df['uninsured_rate'])       # high uninsured = bad
df['drive_score']      = minmax(df['estimated_drive_minutes'])  # long drive = bad
df['physician_score']  = 1 - minmax(df['pcp_per_100k'])    # low physicians = bad, so invert

print('Normalization check (should all be 0.0 to 1.0):')
for col in ['poverty_score', 'uninsured_score', 'drive_score', 'physician_score']:
    print(f'  {col}: min={df[col].min():.3f}, max={df[col].max():.3f}, mean={df[col].mean():.3f}')


## Step 4: Compute Composite Desert Score

We combine the four normalized components using evidence-based weights
drawn from HRSA's Health Professional Shortage Area (HPSA) methodology
and rural health access literature:

| Component | Weight | Rationale |
|---|---|---|
| Drive time to nearest ED | 35% | Geographic access is the primary structural barrier — if you can't physically reach care, nothing else matters |
| Physician supply | 30% | Provider shortage is HRSA's primary HPSA criterion; defines whether primary care exists in the community |
| Uninsured rate | 20% | Financial access — uninsured patients delay or avoid care regardless of proximity |
| Poverty rate | 15% | Socioeconomic barrier — correlated with uninsurance but captures broader economic stress |

These weights are defined as a config dict, not hardcoded, so you can
adjust them and rerun to see how sensitive the results are to weighting choices.

**Methodological note:** Arbitrary weighting is a known limitation of composite
indexes. Alternatives include PCA (let the data determine weights) and the Delphi
method (expert consensus). We use literature-based weights here for transparency
and interpretability — a public audience can understand '35% geographic access'
more easily than eigenvalues.

In [ ]:
# Weights are configurable — adjust and rerun to test sensitivity
WEIGHTS = {
    'drive_score':     0.35,  # geographic access
    'physician_score': 0.30,  # provider supply
    'uninsured_score': 0.20,  # financial access
    'poverty_score':   0.15,  # socioeconomic barrier
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, 'Weights must sum to 1.0'

# Compute weighted composite Desert Score
df['desert_score'] = sum(
    df[col] * weight for col, weight in WEIGHTS.items()
)

print('Desert Score distribution:')
print(df['desert_score'].describe().round(3))
print()
print('Top 5 worst-access tracts:')
print(df.nlargest(5, 'desert_score')[['tract_geoid', 'county_fips', 'desert_score',
                                        'drive_score', 'physician_score']].to_string(index=False))


## Step 5: Classify Tracts into Desert Tiers

We classify tracts into four tiers using **quartiles** of the Desert Score.
Quartile-based classification is preferable to fixed cutoffs because it's
self-calibrating — the tiers always reflect Colorado's internal distribution
rather than an arbitrary national threshold that may not apply here.

| Tier | Quartile | Label |
|---|---|---|
| 1 | Top 25% (worst) | Severe Desert |
| 2 | 50th–75th percentile | Moderate Desert |
| 3 | 25th–50th percentile | Underserved |
| 4 | Bottom 25% (best) | Adequate Access |

In [ ]:
q25 = df['desert_score'].quantile(0.25)
q50 = df['desert_score'].quantile(0.50)
q75 = df['desert_score'].quantile(0.75)

def assign_tier(score):
    if score >= q75:
        return 'Severe Desert'
    elif score >= q50:
        return 'Moderate Desert'
    elif score >= q25:
        return 'Underserved'
    else:
        return 'Adequate Access'

df['desert_tier'] = df['desert_score'].apply(assign_tier)

print('Tier distribution:')
print(df['desert_tier'].value_counts())
print()

# Quick bar chart
tier_order = ['Severe Desert', 'Moderate Desert', 'Underserved', 'Adequate Access']
tier_colors = ['#7b2d8b', '#c46db0', '#f4a9d1', '#d4edda']
counts = df['desert_tier'].value_counts().reindex(tier_order)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(tier_order, counts.values, color=tier_colors, edgecolor='white')
ax.set_title('Colorado Census Tracts by Desert Tier', fontsize=14)
ax.set_ylabel('Number of Tracts')
ax.set_xlabel('Desert Tier')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', fontsize=11)
plt.tight_layout()
plt.show()
print('\nInterpretation: Each tier contains roughly 25% of Colorado\'s 1,447 census tracts.')


## Step 6: Validate Against CDC PLACES Health Outcomes

The Desert Index is our *input* measure of access. CDC PLACES gives us
*outcome* measures — actual health status of people in each tract.

If the Desert Index is measuring something real, tracts with higher desert
scores should show worse health outcomes. We test this using **Spearman
rank correlation** rather than Pearson because:

1. Our desert score is a composite index, not normally distributed
2. Spearman is robust to outliers — a few extreme rural tracts won't distort it
3. We care about *rank order* (is worse access associated with worse outcomes?),
   not the linear magnitude of the relationship

**What correlation tells us and what it doesn't:**
A significant correlation confirms the index has construct validity — it's
measuring something that connects to real health. It does *not* mean poor
access *causes* poor health. Both could be driven by a third factor
(poverty, demographics, historical disinvestment). This is observational
geographic data — causal claims require much stronger designs.

In [ ]:
from scipy import stats

# Load CDC PLACES outcomes for Colorado
cur.execute("""
    SELECT tractfips, measure, data_value
    FROM health_clean.cdc_places_clean
    WHERE statefips = '08'
      AND geo_level = 'Census Tract'
      AND measure IN (
          'Diagnosed diabetes among adults',
          'Current lack of health insurance among adults',
          'Visits to doctor for routine checkup within the past year among adults'
      )
""")
rows = cur.fetchall()
df_places = pd.DataFrame(rows, columns=['tract_geoid', 'measure', 'data_value'])

# Pivot to wide format: one row per tract, one column per measure
df_places_wide = df_places.pivot_table(
    index='tract_geoid', columns='measure', values='data_value'
).reset_index()
df_places_wide.columns.name = None

# Rename for clarity
df_places_wide.columns = [
    c.replace('Diagnosed diabetes among adults', 'diabetes_pct')
     .replace('Current lack of health insurance among adults', 'uninsured_pct')
     .replace('Visits to doctor for routine checkup within the past year among adults', 'checkup_pct')
    for c in df_places_wide.columns
]

# Join Desert Score to outcomes
df_val = df[['tract_geoid', 'desert_score', 'desert_tier']].merge(
    df_places_wide, on='tract_geoid', how='inner'
)
print(f'Matched tracts for validation: {len(df_val)}')
print(df_val.head())


In [ ]:
# Spearman correlation: Desert Score vs each health outcome
outcomes = [c for c in df_val.columns if c not in ['tract_geoid', 'desert_score', 'desert_tier']]

print('Spearman Correlation: Desert Score vs Health Outcomes')
print('=' * 55)
for outcome in outcomes:
    clean = df_val[['desert_score', outcome]].dropna()
    if len(clean) < 10:
        continue
    rho, pval = stats.spearmanr(clean['desert_score'], clean[outcome])
    sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ''))
    print(f'  {outcome[:45]:<45} rho={rho:+.3f}  p={pval:.4f} {sig}')

print()
print('Interpretation:')
print('  Positive rho = higher desert score associates with higher outcome rate')
print('  Negative rho = higher desert score associates with lower outcome rate')
print('  *** p<0.001, ** p<0.01, * p<0.05')


## Step 7: Write Results to health_clean.desert_index

Now that we've validated the index, we write the final tract-level scores
to `health_clean.desert_index`. This is the table the Dash app will query
directly to color the choropleth map.

In [ ]:
from datetime import datetime, timezone

# Prepare final output columns
df_out = df[[
    'tract_geoid', 'county_fips', 'total_population',
    'poverty_rate', 'uninsured_rate', 'pcp_per_100k', 'estimated_drive_minutes',
    'poverty_score', 'uninsured_score', 'physician_score', 'drive_score',
    'desert_score', 'desert_tier'
]].copy()

# Add state_code and weight config for reproducibility
df_out['state_code'] = 'CO'
df_out['weight_drive_time']  = WEIGHTS['drive_score']
df_out['weight_physician']   = WEIGHTS['physician_score']
df_out['weight_uninsured']   = WEIGHTS['uninsured_score']
df_out['weight_poverty']     = WEIGHTS['poverty_score']
df_out['computed_at']        = datetime.now(timezone.utc)

# Clear existing CO data and reinsert
cur.execute("DELETE FROM health_clean.desert_index WHERE state_code = 'CO'")
deleted = cur.rowcount
print(f'Deleted {deleted} existing rows')

insert_sql = """
    INSERT INTO health_clean.desert_index (
        tract_geoid, county_fips, total_population,
        poverty_rate, uninsured_rate, physician_ratio, drive_time_minutes,
        poverty_score, uninsured_score, physician_score, drive_time_score,
        desert_score, desert_tier, state_code,
        weight_drive_time, weight_physician, weight_uninsured, weight_poverty,
        computed_at
    ) VALUES (
        %s, %s, %s,
        %s, %s, %s, %s,
        %s, %s, %s, %s,
        %s, %s, %s,
        %s, %s, %s, %s,
        %s
    )
"""

batch = []
for _, row in df_out.iterrows():
    batch.append((
        row['tract_geoid'], row['county_fips'], int(row['total_population']) if pd.notna(row['total_population']) else None,
        float(row['poverty_rate']), float(row['uninsured_rate']),
        float(row['pcp_per_100k']), float(row['estimated_drive_minutes']),
        float(row['poverty_score']), float(row['uninsured_score']),
        float(row['physician_score']), float(row['drive_score']),
        float(row['desert_score']), row['desert_tier'], 'CO',
        row['weight_drive_time'], row['weight_physician'],
        row['weight_uninsured'], row['weight_poverty'],
        row['computed_at']
    ))

cur.executemany(insert_sql, batch)
conn.commit()
print(f'Inserted {len(batch)} tracts into health_clean.desert_index')
print()
print('Sample output:')
print(df_out[['tract_geoid', 'desert_score', 'desert_tier']].head(10).to_string(index=False))


## Summary

The Colorado Healthcare Desert Index is now computed and stored. Each of
Colorado's 1,447 census tracts has been scored on four access dimensions,
combined into a composite Desert Score, and classified into one of four tiers.

**Key findings to explore in the Dash app:**
- Which counties have the most Severe Desert tracts?
- Do hospitals cluster in Adequate Access areas, leaving Severe Desert tracts with no nearby care?
- Are non-compliant hospitals more likely to serve Severe Desert communities?

**Known limitations:**
- Physician supply is county-level — all tracts in a county share the same ratio
- Drive time is estimated via Haversine (straight-line × 1.4), not actual road routing
- Weights are literature-based, not empirically derived for Colorado specifically

**Next:** `app/app.py` — the Plotly Dash application.